# RAG Chatbot Evaluation — Colab (Llama-3.1-8B, Ollama / vLLM / HF)

Runs the chatbot accuracy eval **off Groq's rate-limited free tier**, so the
judge LLM never gets `429`-throttled mid-run. Accuracy, not speed.

**Metrics (5):**
- `ContextRecall`, `Faithfulness`, `AnswerRelevancy` — ragas (LLM-judged)
- `SemanticSimilarity` — *semantic*: mpnet cosine of answer vs ground truth (added)
- `RougeL` — *lexical*: ROUGE-L F1 of answer vs ground truth (added)

**Models:** the **generator under test** is `meta-llama/Llama-3.1-8B-Instruct` (fp16) — the open-weights
model behind Groq's `llama-3.1-8b-instant`, matching production. The ragas **judge** is a *stronger*
local model (`gpt-oss:20b`, set in CONFIG): an 8B judge schema-echoes `AnswerRelevancy` to `nan`, so the
judge is decoupled from the generator and run locally (free, no Groq daily-token cap).

**Outputs (written to `eval/`):** per-sample scores (`rag_results_colab.json` + `.csv`),
aggregate means (`rag_aggregate_colab.csv`), and a before/after table (`rag_before_after_colab.csv`).
Just run top to bottom.

**Pick an engine in the CONFIG cell:**
- `ollama` — local server on the Colab GPU. **Recommended (free).** One self-contained binary,
  no CUDA-wheel matching, OpenAI-compatible. On an A100 40GB it pulls the **fp16** weights (~16GB) —
  full precision, so judging quality matches Groq; no quantization loss.
- `vllm` — local server on the Colab GPU. Free and fastest, but the prebuilt vLLM wheel often
  mismatches Colab's CUDA (`libcudart.so.*` ImportError); if it won't install cleanly, use `ollama`.
- `hf_api` — HF Inference Providers. Real fp16, no GPU build, but **needs paid inference credits**
  (the free monthly allowance runs out fast).

> **Clean A/B note:** the committed baseline was judged by *Groq*. For a perfectly controlled
> before/after of the chunking fix, run this notebook twice with the **same** engine: once on the
> cloned (old) KB, once on your rebuilt KB (upload cell). Cross-engine deltas are only *directional*.

## 1. Install dependencies

In [ ]:
%pip install -q "ragas==0.4.3" openai sentence-transformers rouge-score \
                onnxruntime tokenizers langchain-community groq huggingface_hub
# The LLM engine itself is set up later, only for the engine you picked:
#   ollama -> installed via curl in the engine cell (recommended free-GPU path)
#   vllm   -> pip-installed in the engine cell (free GPU, but CUDA-wheel fragile on Colab)

## 2. Get the repo (retrieval engine, prompt, ground truth, KB)

In [ ]:
import os
if not os.path.isdir("pose_est_v2"):
    !git clone -q -b test https://github.com/cemmacabales/pose_est_v2.git
%cd pose_est_v2
print("KB on disk:", os.path.getsize("data/knowledge_base.json"), "bytes")

### 2b. (Optional) Upload your rebuilt KB

The clone carries whatever `data/knowledge_base.json` is pushed on the `test` branch.
If your chunking-fix KB **isn't pushed yet**, run this cell and pick your local
`data/knowledge_base.json` to evaluate the fix. Skip otherwise.

In [ ]:
from google.colab import files
import shutil
up = files.upload()
if up:
    name = next(iter(up))
    shutil.move(name, "data/knowledge_base.json")
    print("Replaced data/knowledge_base.json with", name)

## 3. Config

In [ ]:
ENGINE      = "ollama"     # "ollama" (free Colab GPU, fp16 — recommended) | "vllm" (free GPU, bf16, install-fragile) | "hf_api" (paid credits)
MODEL       = "meta-llama/Llama-3.1-8B-Instruct"   # used by hf_api only; ollama/vllm set their own MODEL_ID in the engine cell
TOP_K       = 6            # system default after the 4->6 retrieval bump (baseline used 4)
MAX_TOKENS  = 512
JUDGE_MODEL = "gpt-oss:20b"  # ragas JUDGE, run locally on the Colab GPU (ollama). Deliberately
                             # stronger than the 8B generator: an 8B judge parrots the JSON schema
                             # back to instructor instead of an instance, so AnswerRelevancy -> nan
                             # and ContextRecall is unstable. Local + free dodges Groq's 100k-tok/day
                             # cap. ~14GB, sits resident next to the fp16 8B generator (~30GB total on
                             # an A100 40GB). Alt if OOM/quirky: "qwen2.5:32b-instruct" (heavier) or
                             # drop the generator to "llama3.1:8b" (Q4). Only used by the ollama engine.
RESULTS_OUT = "eval/rag_results_colab.json"
BASELINE    = "eval/rag_results_source_aware.json"  # committed pre-fix scores (Groq-judged)

## 4. Start the LLM engine

Both branches end up exposing the same OpenAI-compatible `(BASE_URL, API_KEY, MODEL_ID)`,
so generation and judging are identical downstream regardless of engine.

In [ ]:
import os, subprocess, time, urllib.request

def _sh(cmd):
    """Run a shell command, raising with the captured output so failures are diagnosable."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"`{cmd}` failed (exit {r.returncode}):\n"
                           f"--- stdout ---\n{r.stdout[-2000:]}\n--- stderr ---\n{r.stderr[-2000:]}")
    return r.stdout

if ENGINE == "hf_api":
    from getpass import getpass
    tok = os.environ.get("HF_TOKEN") or getpass("HF token (needs Llama-3.1 access + inference credits): ")
    os.environ["HF_TOKEN"] = tok
    BASE_URL, API_KEY, MODEL_ID = "https://router.huggingface.co/v1", tok, MODEL
    # If the router can't auto-pick a provider, pin one, e.g. MODEL + ":fireworks-ai" or ":together".
    print("HF Inference Providers ->", MODEL_ID)

elif ENGINE == "ollama":
    # Free local inference on the Colab GPU. Install via the official release *tarball* (not
    # install.sh, which sets up a systemd service / NVIDIA drivers and exits 1 in a container).
    # As of v0.30 the asset is `ollama-linux-amd64.tar.zst` (zstd, on GitHub releases) and bundles
    # the CUDA libs, so it's self-contained — no Python/CUDA-wheel matching. OpenAI-compatible at
    # :11434/v1, so everything downstream is unchanged. On the A100 40GB we pull the *fp16* weights
    # (~16GB) — full precision, matching Groq's fp16; no 4-bit quantization loss.
    # (Drop OLLAMA_MODEL to "llama3.1:8b" for the 4-bit Q4 build if you're on a smaller GPU.)
    if not os.path.exists("/usr/bin/ollama") and not os.path.exists("/usr/local/bin/ollama"):
        print("Installing Ollama (release tarball)...")
        _sh("command -v zstd >/dev/null 2>&1 || (apt-get -qq update && apt-get -qq install -y zstd)")
        _sh("curl -fSL https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tar.zst "
            "-o /tmp/ollama.tar.zst")
        _sh("tar -C /usr --zstd -xf /tmp/ollama.tar.zst")
    srv = subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if srv.poll() is not None:
            print("\n----- ollama.log (tail) -----\n" + open("ollama.log").read()[-3000:])
            raise RuntimeError("`ollama serve` exited — see log above (is the runtime on a GPU?).")
        try:
            urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2); break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("ollama serve did not come up in time — check ollama.log")
    MODEL_ID = os.environ.get("OLLAMA_MODEL", "llama3.1:8b-instruct-fp16")  # ~16GB fp16, fits A100 40GB
    print(f"Pulling {MODEL_ID} (first run downloads weights; ~16GB at fp16)...")
    _sh(f"ollama pull {MODEL_ID}")
    BASE_URL, API_KEY = "http://localhost:11434/v1", "ollama"
    # Pull the judge too — it stays resident alongside the generator, so the
    # gen/judge interleaving in the eval loop never has to swap models.
    print(f"Pulling judge {JUDGE_MODEL} (local, ~14GB; resident with the 8B generator)...")
    _sh(f"ollama pull {JUDGE_MODEL}")
    JUDGE_BASE_URL, JUDGE_API_KEY, JUDGE_MODEL_ID = BASE_URL, API_KEY, JUDGE_MODEL
    print("Ollama ready -> gen", MODEL_ID, "| judge", JUDGE_MODEL_ID)

elif ENGINE == "vllm":
    import torch
    subprocess.run(["pip", "install", "-q", "vllm", "bitsandbytes"], check=True)
    # Default to the UNGATED mirror so no HF token/gating is needed — same weights
    # as the gated meta-llama/Llama-3.1-8B-Instruct (Groq's model). If you have an
    # HF token with Llama access, set HF_TOKEN and VLLM_MODEL to the gated id.
    VLLM_MODEL = os.environ.get("VLLM_MODEL", "NousResearch/Meta-Llama-3.1-8B-Instruct")
    if os.environ.get("HF_TOKEN"):
        from huggingface_hub import login; login(os.environ["HF_TOKEN"])
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    quant = [] if vram >= 22 else ["--quantization", "bitsandbytes", "--load-format", "bitsandbytes"]
    print(f"GPU ~{vram:.0f}GB -> {'bf16 (closest to Groq)' if not quant else '4-bit bitsandbytes (fits T4)'} | model={VLLM_MODEL}")
    MODEL_ID = "llama-3.1-8b-instruct"
    cmd = ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", VLLM_MODEL,
           "--served-model-name", MODEL_ID, "--dtype", "bfloat16", "--max-model-len", "8192",
           "--gpu-memory-utilization", "0.92",
           "--enable-auto-tool-choice", "--tool-call-parser", "llama3_json"] + quant
    srv = subprocess.Popen(cmd, stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
    BASE_URL, API_KEY = "http://localhost:8000/v1", "EMPTY"
    print("Starting vLLM (first run downloads weights; ~3-6 min)... tail vllm.log to watch.")
    for _ in range(240):
        if srv.poll() is not None:   # process died — surface the log instead of spinning
            print("\n----- vllm.log (tail) -----\n" + open("vllm.log").read()[-3000:])
            raise RuntimeError(
                "vLLM exited early — see log above. Common causes: CUDA/torch mismatch on the "
                "Colab image (e.g. 'libcudart.so.13: cannot open shared object file'), OOM, or a "
                "gated model. If vLLM won't install cleanly, set ENGINE='ollama' (no CUDA-wheel "
                "build; runs full-precision fp16 Llama-3.1-8B on the same GPU).")
        try:
            urllib.request.urlopen("http://localhost:8000/health", timeout=2); print("vLLM ready."); break
        except Exception:
            time.sleep(5)
    else:
        raise RuntimeError("vLLM did not come up in time — check vllm.log")
else:
    raise ValueError(f"Unknown ENGINE: {ENGINE}")

# Engines without a second local model (vllm, hf_api) judge on the generator
# path (the 8B-judge schema-echo caveat applies there). ollama set these above.
JUDGE_BASE_URL = locals().get("JUDGE_BASE_URL", BASE_URL)
JUDGE_API_KEY  = locals().get("JUDGE_API_KEY", API_KEY)
JUDGE_MODEL_ID = locals().get("JUDGE_MODEL_ID", MODEL_ID)


## 5. Retrieval + faithful chatbot generation

Reuses the repo's real `RetrievalEngine` and `build_system_prompt`, so generated
answers match the deployed chatbot. (`session_chat.llm` requires a `GROQ_API_KEY`
at import time — we set a dummy; no Groq call is ever made here.)

In [ ]:
os.environ.setdefault("GROQ_API_KEY", "unused-on-colab")  # satisfy session_chat.llm import only
from openai import OpenAI, AsyncOpenAI
from session_chat.retrieval import RetrievalEngine
from session_chat.llm import build_system_prompt

gen_client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
retriever  = RetrievalEngine(kb_path="data/knowledge_base.json", model_dir="data/embedding_model")

# Same session stub eval_rag.py uses, so session-specific questions (e.g. sets/reps)
# aren't refused for lack of an exercise list.
SESSION_STUB = {
    "date": "2026-01-15", "duration_seconds": 1800, "overall_form_score_pct": 71,
    "total_exercises_detected": 5,
    "exercises": [
        {"name": "Deep Squat", "duration_seconds": 240, "form_score_pct": 72},
        {"name": "Hurdle Step", "duration_seconds": 210, "form_score_pct": 65},
        {"name": "Inline Lunge", "duration_seconds": 240, "form_score_pct": 70},
        {"name": "Standing Leg Raise", "duration_seconds": 180, "form_score_pct": 68},
        {"name": "Side Lunge", "duration_seconds": 200, "form_score_pct": 75},
    ],
}

def generate(query, contexts):
    chunks = [{"text": c, "source": "knowledge_base", "page": "?", "section_title": ""} for c in contexts]
    sysp = build_system_prompt(SESSION_STUB, chunks)
    r = gen_client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "system", "content": sysp}, {"role": "user", "content": query}],
        temperature=0.0, max_tokens=MAX_TOKENS)
    return r.choices[0].message.content

## 6. ragas metrics (3) — same setup as `eval/eval_rag.py`

In [ ]:
import sys
from types import ModuleType
# ragas hard-imports a vertexai chat model removed from langchain-community 0.3+; stub it out.
_k = "langchain_community.chat_models.vertexai"
if _k not in sys.modules:
    _m = ModuleType(_k); _m.ChatVertexAI = type("ChatVertexAI", (), {}); sys.modules[_k] = _m

from ragas.metrics.collections import AnswerRelevancy, ContextRecall, Faithfulness
from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings

# Judge on the stronger local model (JUDGE_MODEL_ID), NOT the 8B generator, so
# structured-output metrics (AnswerRelevancy especially) don't schema-echo to nan.
judge_client = AsyncOpenAI(base_url=JUDGE_BASE_URL, api_key=JUDGE_API_KEY)
ragas_llm = llm_factory(JUDGE_MODEL_ID, client=judge_client)
ragas_emb = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2",
                                  use_api=False, normalize_embeddings=True)
RAGAS_METRICS = [ContextRecall(llm=ragas_llm),
                 Faithfulness(llm=ragas_llm),
                 AnswerRelevancy(llm=ragas_llm, embeddings=ragas_emb)]
print("ragas metrics:", [m.__class__.__name__ for m in RAGAS_METRICS], "| judge:", JUDGE_MODEL_ID)

## 7. The 2 added metrics (reference-based, deterministic, no LLM)

- **Semantic** — `SemanticSimilarity`: cosine of the answer vs the ground-truth answer in
  `all-mpnet-base-v2` space (a stronger STS model than the system's MiniLM, for a more
  accurate semantic read). "Does the answer *mean* the right thing?"
- **Lexical** — `RougeL`: ROUGE-L F1, longest-common-subsequence word overlap of answer vs
  ground truth. "Does it use the right *words*?"

Both are rate-limit-proof, so they always populate even if a judge call fails.

In [ ]:
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer

_sem = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
def semantic_similarity(resp, ref):
    e = _sem.encode([resp, ref], normalize_embeddings=True)
    return float(util.cos_sim(e[0], e[1]))

_rs = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
def rouge_l(resp, ref):
    return float(_rs.score(ref, resp)["rougeL"].fmeasure)

## 8. Run the evaluation

In [ ]:
import json, csv, time, inspect, math

samples = json.load(open("eval/rag_ground_truth.json"))["samples"]
DELAY = 0  # no Groq TPM cap here; raise if your HF provider rate-limits
rows = []
print(f"Evaluating {len(samples)} samples | engine={ENGINE} model={MODEL_ID} top_k={TOP_K}\n")

for i, item in enumerate(samples):
    q, ref = item["query"], item["ground_truth"]
    ctxs = [c["text"] for c in retriever.search(q, top_k=TOP_K)]
    resp = generate(q, ctxs)
    row = {"query": q,
           "SemanticSimilarity": semantic_similarity(resp, ref),
           "RougeL": rouge_l(resp, ref)}
    payload = {"user_input": q, "retrieved_contexts": ctxs, "response": resp, "reference": ref}
    for metric in RAGAS_METRICS:
        name = metric.__class__.__name__
        req = set(inspect.signature(metric.ascore).parameters) - {"self"}
        try:
            # abatch_score (not the sync batch_score): Colab runs inside a live asyncio
            # loop, and ragas refuses the sync path there. `await` works because IPython
            # auto-wraps a cell containing top-level await as a coroutine.
            res = await metric.abatch_score([{k: v for k, v in payload.items() if k in req}])
            val = res[0].value
            row[name] = float(val) if val is not None else float("nan")
        except Exception as e:
            print(f"  [{i}] {name} failed: {e}")
            row[name] = float("nan")
        if DELAY:
            time.sleep(DELAY)
    rows.append(row)
    print(f"  [{i+1:>2}/{len(samples)}] "
          f"CR={row['ContextRecall']:.2f} F={row['Faithfulness']:.2f} AR={row['AnswerRelevancy']:.2f} "
          f"Sem={row['SemanticSimilarity']:.2f} R-L={row['RougeL']:.2f} | {q[:42]}")

ORDER = ["ContextRecall", "Faithfulness", "AnswerRelevancy", "SemanticSimilarity", "RougeL"]
CSV_OUT, AGG_OUT = RESULTS_OUT.replace(".json", ".csv"), "eval/rag_aggregate_colab.csv"

# Per-sample JSON
json.dump(rows, open(RESULTS_OUT, "w"), indent=2, ensure_ascii=False)

# Per-sample CSV
with open(CSV_OUT, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["query"] + ORDER, extrasaction="ignore")
    w.writeheader()
    for r in rows:
        w.writerow(r)

# Aggregate CSV (one row per metric)
def agg(k):
    vs = [r[k] for r in rows if isinstance(r.get(k), (int, float)) and not math.isnan(r[k])]
    return (round(sum(vs) / len(vs), 4) if vs else "", len(vs))
with open(AGG_OUT, "w", newline="") as f:
    w = csv.writer(f); w.writerow(["metric", "mean", "n"])
    for k in ORDER:
        m, n = agg(k); w.writerow([k, m, n])

print(f"\nSaved -> {RESULTS_OUT}\n         {CSV_OUT}  (per-sample)\n         {AGG_OUT}  (aggregate)")
print("\n=== Aggregate ===")
for k in ORDER:
    m, n = agg(k)
    print(f"  {k:18} {m if m != '' else 'n/a'}  (n={n})")

## 9. Before / after vs the committed baseline

In [ ]:
import json, csv, math

base = {r["query"]: r for r in json.load(open(BASELINE))}
new  = {r["query"]: r for r in rows}

def mean(d, k):
    vs = [r[k] for r in d.values() if isinstance(r.get(k), (int, float)) and not math.isnan(r[k])]
    return (sum(vs) / len(vs), len(vs)) if vs else (float("nan"), 0)

ba_rows, RAGAS = [], ["ContextRecall", "Faithfulness", "AnswerRelevancy"]
print(f"{'metric':20}{'baseline':>11}{'colab':>11}{'delta':>9}")
for k in RAGAS:
    (bm, _), (nm, _) = mean(base, k), mean(new, k)
    print(f"{k:20}{bm:>11.3f}{nm:>11.3f}{nm-bm:>+9.3f}")
    ba_rows.append([k, round(bm, 4), round(nm, 4), round(nm - bm, 4)])
for k in ["SemanticSimilarity", "RougeL"]:
    nm, _ = mean(new, k)
    print(f"{k:20}{'-':>11}{nm:>11.3f}{'new':>9}")
    ba_rows.append([k, "", round(nm, 4), "new"])

with open("eval/rag_before_after_colab.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["metric", "baseline", "colab", "delta"]); w.writerows(ba_rows)
print("\nSaved -> eval/rag_before_after_colab.csv")

print(f"\nbaseline judge = Groq llama-3.1-8b-instant | colab judge = {JUDGE_MODEL_ID} (stronger, local).")
print("Baseline also used top_k=4; this run uses top_k=6. Deltas are directional, not a controlled A/B.")
print("For a strict A/B, re-run this notebook on the OLD KB with the same engine.")